# Week 1 Day 2 Lab - Introducing UnderCover (free version)

## LLMs and Agents on a commercial task: insurance claims

A claims handler at UnderCover Travel Insurance has already decided each of the 25 claims in its queue. We send every claim to a free model, along with the customer's policy record and the policy wording. Each model replies with a decision of approve, decline or refer, a rationale and a payout. Then we compare the answers with the handlers' decisions.

At the end we give the same job to a LangChain agent that uses tools to look up the policy and record its decision.

## 3 parts:

PART 1: Examine the data

PART 2: Run with a free model

PART 3: An Agent via LangChain

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This is the free version of the lab. It uses <code>openrouter/free</code> throughout, so it needs no credit on OpenRouter. The full version with paid models is <a href="week1day2.ipynb">week1day2.ipynb</a>.<br/><br/>
            OpenRouter's free models allow 50 requests a day on an account without credit. With <code>TINY = True</code>, the default here, this lab makes about 30. <code>openrouter/free</code> picks a different free model for each request, so expect your results to vary more than mine.<br/><br/>
            Please execute each cell and compare your results with mine. Add print statements and try small adjustments to make the code your own.<br/><br/>
            Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Resources</h2>
            <span style="color:#118AB2;">The <a href="../guides/01_intro.ipynb">guides</a> folder in this repo has more information on topics like the command line, Git, Python and debugging.<br/><br/>
            If this lab is too much, consider courses 1, 2, 4 and 5 in <a href="https://edwarddonner.com/curriculum">my curriculum</a>: AI Builder, AI Coder, and the Core and Agentic Tracks of AI Engineer.<br/><br/>
            If it's too slow, put me on 2x and zip through it!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/connect.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Stay in touch</h2>
            <span style="color:#118AB2;">Please message me in Udemy anytime with questions, or email me at <a href="mailto:ed@edwarddonner.com">ed@edwarddonner.com</a>.<br/><br/>
            Let's connect on <a href="https://www.linkedin.com/in/eddonner/">LinkedIn</a>! And subscribe to my <a href="https://www.youtube.com/@edward.donner">YouTube channel</a> for more on LLMs, agents and coding agents.
            </span>
        </td>
    </tr>
</table>

In [1]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from pydantic import BaseModel, Field

## Connect to OpenRouter - and please read note here

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

### NOTE:

If this is a new OpenRouter account, then you might get an error at some point from OpenRouter like: "new accounts are limited to 20 requests per minute for this model"

If you get that error, then add this line immediately after calling an LLM:

`time.sleep(3)`

and you should be in good shape.

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

OpenRouter API key found


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: the Chat Completions API</h2>
            <span style="color:#118AB2;">We call models with the Chat Completions API. <code>messages</code> is a Python list of dicts, each with a <code>role</code> and <code>content</code>. The system prompt tells the model how to behave and the user prompt has the request. The reply is in <code>response.choices[0].message.content</code>.<br/><br/>
            Here's a quick test.
            </span>
        </td>
    </tr>
</table>

In [3]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = openrouter.chat.completions.create(model="openrouter/free", messages=messages)
print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Answered by xingchen-agi/xing4.0-29b-a4b:free


Aye, the question "What is the meaning of the universe?" is the grandest query any sailor ever set their eyes upon. It is a deep sea mystery, and pulling the answer from those depths requires us to look at it from many angles.

Some sailors look to the stars and the charts of science. They tell us that the universe is a vast, spinning mechanism—born in a violent explosion, made of dust and light, governed by strict, unyielding laws. From this view, the universe doesn't carry a signpost or a golden ticket. It simply *is*. There is no inherent, built-in purpose stamped into the fabric of reality. It just happens, and waits for someone to ask, "Why?"

But then comes the twist in our tale: we are part of that universe. We are made of the same stardust as the galaxies, yet we possess a spark of awareness. Philosopher Blaise Pascal once suggested that the entire universe is like a huge book, and we are the only characters inside who care to read it. So, does the universe have a pre-written script? Most likely not. Meaning is something we project onto the world. We seek it because it is woven into the very fiber of our being. 

Then there are those who look to the divine. They say the universe is a grand canvas painted by a creator, and its meaning lies in serving that creator's ultimate design, whether that be love, justice, or salvation.

So, what is the meaning? Honestly? The universe has no inkblot on its sleeve. It doesn't carry a message just for you. But *you*—with your laughter, your curiosity, your capacity to love, and your brilliant capacity to create—are the universe asking itself why it exists. Whether the answer is to explore, to love, to build, or to simply wonder, that meaning is written in the stars and forged in your own heart. 

Go forth, ye mariner, and chart your own course.

## Part 1: Load and examine the data

The seed data is in the repo's `data` folder. `get_sample_data` returns the claims and the handlers' decisions on them, keyed by claim id. The model is sent the claims, the policies and the policy wording.

The queue has 25 claims. This version sets `TINY = True` to keep 5 of them, which fits the free daily limit. Set it to `False` for all 25 if your account allows more requests.

In [4]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [5]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

25 claims, 25 labels, 50 policies
The policy wording is 1,369 words


In [6]:
claims[0]

{'claim_id': 'CLM-1001',
 'policy_id': 'POL-2001',
 'customer_id': 'CUS-3001',
 'claim_type': 'trip_delay',
 'submitted_on': '2026-06-24',
 'incident_date': '2026-06-18',
 'incident_location': 'Rome, Italy',
 'summary': 'Flight home canceled by the airline, rebooked the next morning',
 'story': 'My flight home from Rome on June 18 was canceled by Meridian Air about two hours before departure because of a crew problem. The next seat they could give me was at 6:55 the following morning. I booked a hotel near the airport, had dinner and breakfast there, and took taxis between the airport and the hotel. I am claiming the hotel, the meals and the taxis, and all of the receipts are attached.',
 'items': [{'description': 'Airport hotel, one night',
   'category': 'accommodation',
   'amount': 245.0},
  {'description': 'Dinner and breakfast', 'category': 'meals', 'amount': 95.0},
  {'description': 'Taxis between the airport and the hotel',
   'category': 'transport',
   'amount': 80.0}],
 'amo

Here is the handler's decision on that claim which we'll use to validate the LLM results.

In [7]:
labels[claims[0]["claim_id"]]

{'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'over_approval_limit': False,
 'handler_id': 'HND-02',
 'decided_on': '2026-06-26',
 'reason': 'The carrier canceled the flight for a crew problem and the delay was 19 hours, well past the 6 hour waiting period on Premium. The hotel, meals and taxis are reasonable and all have receipts. The total is inside the $800 maximum.',
 'reference': 'Policy wording, section 3',
 'calculation': [{'step': 'Eligible expenses with receipts', 'amount': 420.0}],
 'tests': 'A routine trip delay with complete documents',
 'difficulty': 'routine',
 'failure_class': '',
 'needs': [],
 'tone': 'plain',
 'tiny': False}

### Explore the data

It's always good to dig into the data before running experiments.

In [8]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,claim_type,amount_claimed,difficulty,decision,payout
0,CLM-1001,"Flight home canceled by the airline, rebooked ...",trip_delay,420.0,routine,approve,420.0
1,CLM-1002,Suitcase lost on a connecting flight,baggage,1150.0,routine,approve,580.0
2,CLM-1003,"Hospital visit abroad after a fall, fractured ...",emergency_medical,3800.0,routine,approve,3750.0
3,CLM-1004,Checked bag arrived 31 hours late,baggage_delay,186.0,routine,approve,186.0
4,CLM-1005,Passport eaten by a giraffe at breakfast,passport,243.0,routine,approve,243.0
5,CLM-1006,Hit by a falling coconut while reading under a...,emergency_medical,425.0,routine,approve,425.0
6,CLM-1007,Phone dropped in the hotel shower,baggage,1099.0,routine,approve,50.0
7,CLM-1008,Trip to Seattle canceled after a tornado carri...,trip_cancellation,2440.0,routine,approve,2240.0
8,CLM-1009,Camera snatched on the Barcelona metro,baggage,980.0,routine,approve,930.0
9,CLM-1010,Emergency dental treatment in Las Vegas for a ...,emergency_medical,1350.0,routine,decline,0.0


Claim CLM-1015 might be a familiar example...

In [9]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Flew home from Paris on arrival after leaving a child at home**

We flew from Chicago to Paris on June 20 as a family of six. Somewhere over the Atlantic I realized that our youngest son, Kevin, who is eight, was not on the plane. We had overslept, we left for the airport in a rush and the head count in the van was wrong. I took the first available flight home the moment we landed in Paris. Kevin was fine. In his words, he had handled it. I am claiming my one-way fare home and my share of the prepaid hotel nights in Paris that I did not use.

**Handler:** decline $0.00. Trip Interruption pays only for the reasons listed in the wording. Returning home because a child was left behind is not one of them, and nobody was sick or injured.


Here's another one that might ring a bell...

In [10]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Emergency dental treatment in Las Vegas for a missing front tooth**

I was in Las Vegas from June 19 for a friend's bachelor party. On the morning of June 21 I woke up on the floor of our suite missing an upper front tooth. I have no memory of the night before, and neither do my friends. I am a dentist, and I have to say that the extraction is consistent with my own technique. I went to an emergency dental clinic that morning for an exam, an X-ray and a temporary bridge, and their bill and notes are attached. There was also a tiger in the bathroom of the suite, which I mention only for completeness. I am not claiming for anything connected with the tiger.

**Handler:** decline $0.00. The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays.


This chart counts the claims of each type, split by the handler's decision.

In [11]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

Here's a breakdown by difficulty. The policy wording is enough to decide the routine and hard claims. The manual claims depend on the claims manual or the customer's history.

In [12]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

## Part 2: running with a free LLM

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: Structured Outputs</h2>
            <span style="color:#118AB2;">With Structured Outputs the model replies in JSON that follows a schema we give it. We describe the schema as a Pydantic class and pass it to <code>parse()</code> as the <code>response_format</code>. The reply comes back as an object of that class in <code>response.choices[0].message.parsed</code>.
            </span>
        </td>
    </tr>
</table>

Starting by defining a Pydantic class to describe the output so we can use Structured Outputs.

The rationale comes before the decision to ensure the model thinks through the decision before making it.

In [13]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

## The prompt

The system prompt has the instructions and the whole policy wording. The user prompt has the customer's policy record and the claim, both as JSON.

In [14]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

### Calling the LLM

In [15]:
def decide(claim, model):
    """Ask one model to decide one claim."""
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        extra_body={"provider": {"require_parameters": True}},
    )
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "seconds": round(time.time() - start, 1),
    }

In [16]:
first = decide(claims[0], "openrouter/free")
first

{'model': 'openrouter/free',
 'technique': 'LLM',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 300.0,
 'rationale': "The claim is for a trip delay (flight home canceled on June 18, 2026 due to a crew problem). Section 3 of the policy explicitly covers 'canceled service' under the Trip Delay benefit. The customer has provided all required documents (carrier report, hotel receipt, meal receipts, taxi receipts). The total claimed amount is $420.00, but the Trip Delay maximum for the premium plan is $300.00. The deductible does not apply to Trip Delay benefits. Therefore, the claim is covered up to the policy maximum of $300.",
 'seconds': 6.6}

### Now running all the claims

In [17]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, claims, labels):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model}: {right} of {len(results)} right in {seconds:.0f} seconds")
    return results

In [18]:
free_results = run("openrouter/free", claims, labels)

openrouter/free: 16 of 25 right in 495 seconds


## Results by difficulty

This counts the claims the model got right at each level of difficulty. Expect the manual claims to be wrong for now.

In [19]:
scored = pd.DataFrame(free_results)
scored.pivot_table(index="model", columns="difficulty", values="right", aggfunc="sum")

difficulty,hard,manual,routine
model,,,
openrouter/free,3,0,13


## Where a model got it wrong

For each claim a model got wrong, this shows its rationale, the handler's reason and what the claim is there to test. Pass in another set of results to see its misses.

In [20]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(free_results, labels)

**CLM-1010, routine**: approve $1,350.00, where the handler said decline $0.00

- Model: The policy is premium plan: Emergency Medical maximum $250,000, deductible $0. This benefit covers 'necessary treatment of a sudden sickness or injury during your covered trip' (section 7), but excludes injuries from adventure sports unless the Adventure Sports add-on was purchased. The claim is for emergency dental care — an exam, X-ray and temporary bridge — not adventure sports. The incident occurred on June 21, 2026, during the covered trip (June 19–22). However, the clinical notes state the patient had 'heavy drinking through the evening of June 20, smelled of alcohol and was unsteady,' suggesting possible intoxication at the time of the loss. Under section 9 losses while impaired by alcohol are excluded. This creates ambiguity: emergency dental care could be prompted by drink, or it could be unrelated personal injury. The documents are complete and consistent with each other. The most plausible basis for decline under section 9 would be that the loss was incurred while the customer was impaired by alcohol — this could represent either a suboptimal claim decision or genuine exclusion coverage. I will therefore approve but flag this as borderline given the intoxication context.
- Handler: The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays.
- Tests: A general exclusion: a loss while impaired by alcohol, found in the clinical notes


**CLM-1013, routine**: refer $0.00, where the handler said decline $0.00

- Model: The claim is for trip delay under an Economy plan. The delay was 7 hours 5 minutes (09:15 to 16:20 on July 24, 2026), which exceeds the Economy 6-hour waiting period, and the carrier confirms a mechanical issue — a covered cause. Both required documents (carrier delay confirmation and receipts) are attached, and the claim was filed within 90 days. The meal expenses ($64) clearly fall under covered trip-delay costs.

However, the claim also includes a $59 airport lounge day pass. Section 3 lists covered expenses as "meals, accommodation, local transport and rebooking fees." An airport lounge pass does not clearly fall into any of these enumerated categories, and the customer states it was purchased for comfort while waiting. The policy wording does not explicitly settle whether lounge access is a covered trip-delay expense. Because the wording does not resolve this question, the claim should be referred to a colleague for a coverage decision on that item.
- Handler: The delay was 7 hours and 5 minutes. The policy is Economy, where the waiting period for Trip Delay is 12 hours, so the benefit does not apply.
- Tests: A clear decline that depends on the plan: the delay would qualify on Premium or First


**CLM-1018, manual**: approve $330.00, where the handler said refer $0.00

- Model: The claim is for theft of a leather jacket and canvas daypack on June 22, 2026, during the covered trip (June 16–25, 2026) in Florence. Section 5 covers theft of baggage and personal belongings during the covered trip. The items were snatched from the customer's hand, so they were not left unattended. Theft was reported to the Carabinieri within the hour (within 24 hours), and the police report is attached, satisfying Section 5's document requirement for theft. No other policy exclusion applies.

Payout under Section 10 (Premium plan):
- Leather jacket: claimed $420, per-item limit $300 → eligible $300
- Canvas daypack: claimed $80, per-item limit $300 → eligible $80
- Subtotal: $380
- Less deductible ($50 applies once to Baggage and Personal Effects): $330
- Below the $1,500 benefit maximum.

All required documents are attached and the story agrees with the police report.
- Handler: A theft from the person, reported to the police within the hour, with the report on file. The manual asks for proof of value for any item claimed above $100, which is a purchase receipt or a card statement, and the jacket at $420 has none. Document request: proof of what the jacket cost, for example a cash withdrawal record from that morning together with the stallholder's details. The daypack at $80 needs no proof. The claim appears payable at $330 once the proof arrives: the jacket at the $300 limit plus $80, less the $50 deductible.
- Tests: A rule found only in the claims manual: proof of value for items claimed above $100


**CLM-1019, manual**: approve $210.00, where the handler said refer $0.00

- Model: The claim is for Trip Delay under the Premium plan. The delay was 9 hours 20 minutes (scheduled 07:30, actual 16:50), which exceeds the Premium plan's 6-hour waiting period. The cause—crew duty time limit requiring a replacement crew—is a covered "crew problem" under section 3. The incident occurred on the trip departure date (2026-08-14) within the covered trip dates, and the policy was active. The claim was submitted within 90 days. The required documents are attached: the carrier's delay confirmation stating the cause, and receipts for eligible expenses (accommodation $112, meals $58, local transport $40). No deductible applies to Trip Delay under section 1. Under section 10: eligible amount is $210, no carrier or third-party payment to subtract, no deductible, and the result is below the Premium plan maximum of $800.
- Handler: On its own the claim is payable: a 9 hour delay on Premium with receipts. It is the customer's third claim in 12 months, after claims filed in October 2025 and March 2026, so it goes to Special Investigations for review before any payment.
- Tests: A rule found only in the claims manual: the third claim in 12 months. It needs the customer's claim history


**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: The claim is for stolen personal items (prescription sunglasses) under the Baggage and Personal Effects benefit. The policy requires that theft be reported to local police within 24 hours and that a copy of the police report be provided. The claimant has provided a letter from the tour guide but no police report. Without the required police report, the claim cannot be processed. This is a case where a colleague needs to review the missing document before deciding.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1022, manual**: refer $0.00, where the handler said approve $155.00

- Model: Claim meets cover: First plan (3‑hour waiting period), trip dates include 20 July 2026, carrier confirms a 30‑hour Belgrade delay due to severe weather/flash flooding (covered cause), and the delay confirmation is attached. However, Section 3 requires receipts for meal and other delay expenses, and the $140 meal claim has no receipts. Under Section 11 the claim waits until the missing document arrives. If receipts were provided, the payable amount would be $140 meals + $85 rebooking = $225, within the First plan $1,500 trip‑delay maximum and with no deductible applying to trip delay.
- Handler: Severe weather held the train for 30 hours 15 minutes, past the 3 hour waiting period on First, and the railway confirms the times and the cause. The rebooking fee has a receipt. The meals have none, so the manual's flat allowance applies: $35 for one traveler for each 24 hours of delay or part of 24 hours, and 30 hours is two days. The incident on board does not change the cause of the delay.
- Tests: A rule found only in the claims manual: the meal allowance without receipts, over a delay of more than one day


**CLM-1023, hard**: approve $61.00, where the handler said refer $0.00

- Model: The claim is for a Trip Delay under Section 3. The policy is the First plan, which has a 3-hour waiting period and a $1,500 Trip Delay maximum (Section 1). The Northwind Airlines delay certificate confirms Flight NW 418 from Dallas to Vancouver on July 3, 2026 was scheduled to depart at 08:15 and actually departed at 17:32 — a delay of over 9 hours, well past the 3-hour waiting period. The certificate states the cause was a fuel pump fault, which is a mechanical problem, a covered reason under Section 3. The incident occurred on the trip's departure date (July 3, 2026), within the covered trip dates, and both travelers (Stephanie Marsh and Colin Marsh) are named on the policy record. The required documents are attached: the carrier's delay confirmation and itemized meal receipts totaling $61. The story and documents agree. Under Section 1, the deductible applies only to Baggage and Personal Effects and Emergency Medical, not to Trip Delay, so no deductible is subtracted. Applying Section 10: eligible amount $61.00, minus $0 from other parties, minus $0 deductible = $61.00, which is below the $1,500 plan maximum.
- Handler: The delay of 9 hours 17 minutes is well past the 3 hour waiting period on First and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the fuel pump fault rests on the customer's account alone. Document request: Northwind's confirmation of the reason for the delay. The claim appears payable at $61 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


**CLM-1024, manual**: approve $250.00, where the handler said refer $0.00

- Model: The claim is covered under the Baggage and Personal Effects benefit for theft during the covered trip. The police report confirms the theft was reported within 24 hours, and the purchase receipt provides proof of value. However, the per-item limit for the premium plan is $300, so the eligible amount is $300. After applying the $50 deductible, the payout is $250. All required documents are attached, and the claim was filed within 90 days of the incident.
- Handler: On its own the claim is payable at $250: a theft from the person, reported the same day, with the phone at the $300 limit less the $50 deductible. The customer's history shows claim CLM-0987 for the same theft on the same date, filed on July 16, approved and paid at $250. The same loss claimed twice goes to Special Investigations, with no comment to the customer beyond saying the claim is being reviewed. It may well be an honest mistake, since she says she was not sure the first claim went through.
- Tests: The customer's claim history: the same loss already claimed and paid. Nothing in the claim itself shows it


**CLM-1025, hard**: approve $71.00, where the handler said refer $0.00

- Model: The claim is for a trip delay. The policy covers trip delay under section 3 when there is a mechanical or crew problem, cancellation of service, severe weather or strike. The carrier (Skylark Airways) has confirmed the delay and its cause (the first officer ran out of duty hours, with a replacement being flown up from Seattle). This is a carrier-caused delay due to a crew problem. The waiting period for economy is 6 hours, and the 14-hour delay exceeds that. However, the claim also includes meal expenses of $71 ($29 for lunch and $42 for dinner). The policy explicitly states we pay 'reasonable extra costs for meals, accommodation, local transport and rebooking fees during the delay, up to your plan's maximum' under section 3. The benefit maximum for trip delay in economy is $300. The claimed meals appear to be reasonable airline-terminal meals. I need to calculate the payout following section 10. Step 1: Start with eligible amounts — meals: $29 + $42 = $71. Step 2: No carrier or other party paid anything toward these meals, so no offsetting amounts. Step 3: Deductible doesn't apply to trip delay. Step 4: Pay $71, up to $300 maximum. So the payout is $71.
- Handler: The delay of 14 hours 15 minutes is past the 12 hour waiting period on Economy and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the crew problem rests on the customer's account alone. Document request: Skylark's confirmation of the reason for the delay. The claim appears payable at $71 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Accuracy, cost and speed are the numbers a customer will ask about. Cost per claim is the easiest one to relate to the business: compare it with what a handler's time on a claim costs.<br/><br/>
            The kind of mistake matters too. A wrong approval pays out money the policy doesn't owe, a wrong decline lets a customer down and a wrong referral costs a handler's time.
            </span>
        </td>
    </tr>
</table>

## Part 3: Using LangChain create_agent

Our approaches so far have used a single LLM call with Structured Outputs.

Now we'll use a different approach: an AI agent.

An LLM, with tools, in a loop, to achieve a goal!

We'll use LangChain's `create_agent` to run the loop for us.

We give it 2 tools. `get_policy` looks up the customer's policy record. `record_decision` saves the decision.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: LangChain create_agent</h2>
            <span style="color:#118AB2;">1. Describe the LLM with a LangChain class like <code>ChatOpenAI</code><br/>
            2. Turn a function into a tool with <code>@tool</code><br/>
            3. <code>create_agent</code> builds the agent and <code>invoke()</code> runs its loop<br/><br/>
            Behind the scenes, a LangGraph graph is created; we'll find out more in Week 2.
            </span>
        </td>
    </tr>
</table>

In [21]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = llm.invoke(messages)
display(Markdown(response.content))

Arrr, ye ask the mightiest question a scallywag can ask, eh, matey?! *Spits over the side of the ship* 

Well, listen close, for I be the keeper of ancient pirate wisdom! The meanin' of the whole blasted universe, the great big cosmic ocean... it ain't bein' written in no star charts, that's for sure!

**The First Answer:** The meanin' be the **BOOTY!** The gold, the jewels, the endless grog! Ye find the treasure, ye conquer the seven seas, and the universe gives a mighty cheer! That's the whole point of creation, I reckon!

**The Second Answer:** But if ye be askin' for a more... *deep* philosophical answer, then the meanin' be the **ADVENTURE** itself! The whole point of the universe be to sail the endless blue, to fight the Kraken, to discover new islands, and to have a grand tale to tell in the tavern! The meanin' ain't the destination, it's the fightin' and the sailin'!

**The Final Answer:** Arrr, but here's the secret the stars don't want ye to know: The meanin' of the universe be **ME!** Ye! The captain! Ye be the meaning! 

So stop yer blubberin' and get to the point! Set sail, find yer treasure, and make yer own meaning! Avast! Yo ho ho! And watch out for the sea monsters! *Laughs and swashbuckles*

In [22]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

The agent uses `openrouter/free` through LangChain's `ChatOpenAI`, pointed at OpenRouter. Its system prompt has the same instructions and policy wording as before, plus a line about the tools. Once the agent has looked up the policy it has the same information as before, so we can compare its score with the single calls in Part 2.

In [23]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

First, trying out one claim. The model asks for the policy, the tool returns it, the model records its decision and then finishes with a short message.

In [24]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_policy (call-cecdd7e4-f287-4e76-8e96-a77209105d7e)
 Call ID: call-cecdd7e4-f287-4e76-8e96-a77209105d7e
  Args:
    policy_id: POL-2001
================================= Tool Message =================================
Name: get_policy

{
  "policy_id": "POL-2001",
  "customer_id": "CUS-3001",
  "plan": "premium",
  "add_ons": [],
  "purchased_on": "2026-04-22",
  "sold_through": "Meridian Air checkout",
  "trip": {
    "destination": "Rome, Italy",
    "departs_on": "2026-06-10",
    "returns_on": "2026-06-18",
    "trip_cost": 3100.0
  },
  "travelers": [
    "Maya Patel"
  ],
  "status": "active"
}
================================== Ai Message ==================================
Tool Calls:
  record_decision (call-f5f6e28f-0806-49ec-a673-97979c514147)
 Call ID: call-f5f6e28f-0806-49ec-a673-97979c514147
  Args:
    payout: 420
    claim_id: CLM-1001
    rationale: The policy is a Premium pl

Now all the claims. Each claim takes three model calls: one to ask for the policy, one to record the decision and one to finish. A claim with no recorded decision counts as wrong.

In [25]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    return {
        "model": "openrouter/free",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1793923200000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.', 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 f

In [28]:
pd.DataFrame(free_results + agent_results).groupby(["model", "technique"], sort=False)[["right", "seconds"]].sum()

right  seconds
model           technique                
openrouter/free LLM           16    494.9
                Agent          7    411.4

In [26]:
show_misses(agent_results, labels)

**CLM-1006, routine**: decline $0.00, where the handler said approve $425.00

- Model: The emergency medical benefit covers sudden sickness or injury during the trip, but the exclusions in section 7 exclude "treatment that can reasonably wait until you are home." The treating doctor at Bentota Bay Medical Centre advised rest and pain relief, indicating the treatment (prescription painkillers, ~$45) could be self-managed at home once the traveler returned. Since the actual treatment that could reasonably wait until home is present, the claim falls under this exclusion. Payout is $0.
- Handler: A sudden injury during the covered trip, confirmed by the clinic and the hotel. The treatment was necessary and modest. The deductible on First is $0. This is the customer's second claim in 12 months, which does not trigger a review.
- Tests: A routine medical claim from a customer with another claim in the queue


**CLM-1009, routine**: approve $980.00, where the handler said approve $930.00

- Model: Baggage and Personal Effects claim approved. The theft occurred within the covered trip dates (July 18–26) in Barcelona, with police report filed within 24 hours and purchase receipt attached. Using section 10: eligible amount $980 (single item; per-item limit with Electronics Plus add-on is $1,200, lower applies); no carrier offset; $0 deductible for Premium plan baggage benefit; maximum $3,000 applies — payout $980.
- Handler: Theft from the person, reported to the police the same day with the report attached. The policy has Electronics Plus, so the camera and its lens count as one electronics item with a $1,200 limit. The $50 deductible on Premium comes off.
- Tests: An add-on that raises the payout: Electronics Plus


## Putting it all together

Here is every experiment in one table. Speed is the average time for one claim.

In [27]:
rows = []
for results in [free_results, agent_results]:
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

,LLM,technique,accuracy,speed
0,openrouter/free,LLM,16 / 25,19.8 seconds a claim
1,openrouter/free,Agent,7 / 9,45.7 seconds a claim


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">This lab is about experimenting with different LLMs and Agents to understand their accuracy and speed.<br/><br/>
            <code>openrouter/free</code> picks a different free model for each request. The free models on OpenRouter each have an id that ends in <code>:free</code>.<br/><br/>
            Challenge: use a specific free model in place of <code>openrouter/free</code>. Which free model gets the most claims right with the agent?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#069D75;">Congratulations! And if I could disturb you with a quick request:</h2>
            <span style="color:#069D75;">You've finished the first main lab. You tried a free model and an agent on UnderCover's claims, with accuracy and speed side by side.<br/><br/>
            On Day 3 we bring in coding agents and start building the UnderCover platform.<br/><br/>
            <b>I have a quick request for you. If you have a moment to rate the course, I'd hugely appreciate it. It's one of the main ways Udemy decides whether
            to show the course to others, and I'd be super grateful.</b>
            </span>
        </td>
    </tr>
</table>